In [1]:
# =============================================================================
# 🏆 CAPA GOLD - ONE BIG TABLE (OBT)
# Pipeline: GetTalent TP6 - Microsoft Fabric
# Workspace: gt-deuler-dev
# Lakehouse Silver: lh_silver (origen)
# Lakehouse Gold: lh_gold (destino)
#
# CONSIGNA EJERCICIO 4:
#   - Combinar todas las tablas Silver en 1 tabla OBT
#   - Incluir columnas relevantes para consumo
#   - Permitir NULLs (es normal en OBT)
#   - Cantidad registros >= dataset con más registros (120,348)
# =============================================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

# CONFIGURACION SPARK
spark = SparkSession.builder \
    .appName("Pipeline_GetTalent_Gold_OBT") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY") \
    .config("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY") \
    .config("spark.databricks.delta.properties.defaults.autoOptimize.optimizeWrite", "true") \
    .getOrCreate()

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# RUTAS
SILVER_BASE = "abfss://9a547f55-a8f7-4c89-b935-fb3da8226ed0@onelake.dfs.fabric.microsoft.com/4d897e0a-b68c-4840-9910-752862308e5b/Files/silver/shop_carrito"
GOLD_BASE = "abfss://9a547f55-a8f7-4c89-b935-fb3da8226ed0@onelake.dfs.fabric.microsoft.com/902f27e9-dd04-40cf-831a-2610a16fb9a7/Files/gold"

print("="*80)
print("CAPA GOLD - ONE BIG TABLE (OBT)")
print("="*80)
print(f"Origen Silver: {SILVER_BASE}")
print(f"Destino Gold: {GOLD_BASE}")
print(f"\nObjetivo: Combinar 7 tablas Silver en 1 tabla OBT")
print(f"Registros minimos esperados: 120,348 (dataset mas grande)\n")


# =============================================================================
# PASO 1: CARGAR TODAS LAS TABLAS SILVER
# =============================================================================

print("="*80)
print("PASO 1: CARGANDO TABLAS SILVER")
print("="*80)

# 1. SHOP_CARRITO (tabla principal - 120,348 registros)
print("\n1. Cargando: shop_carrito")
df_carrito = spark.read.format("delta").load(f"{SILVER_BASE}/shop_carrito")
count_carrito = df_carrito.count()
print(f"   OK {count_carrito:,} registros")

# 2. SHOP_CLIENTES (115,544 registros)
print("\n2. Cargando: shop_clientes")
df_clientes = spark.read.format("delta").load(f"{SILVER_BASE}/shop_clientes")
count_clientes = df_clientes.count()
print(f"   OK {count_clientes:,} registros")

# 3. ARTICULOS (58 registros)
print("\n3. Cargando: articulos")
df_articulos = spark.read.format("delta").load(f"{SILVER_BASE}/articulos")
count_articulos = df_articulos.count()
print(f"   OK {count_articulos:,} registros")

# 4. CAMPANIAS_GP (96 registros)
print("\n4. Cargando: campanias_gp")
df_campanias = spark.read.format("delta").load(f"{SILVER_BASE}/campanias_gp")
count_campanias = df_campanias.count()
print(f"   OK {count_campanias:,} registros")

# 5. SHOP_CARRITO_ESTADOS (16 registros)
print("\n5. Cargando: shop_carrito_estados")
df_estados = spark.read.format("delta").load(f"{SILVER_BASE}/shop_carrito_estados")
count_estados = df_estados.count()
print(f"   OK {count_estados:,} registros")

# 6. SHOP_CARRITO_ESTADOS_PAGO (5 registros)
print("\n6. Cargando: shop_carrito_estados_pago")
df_estados_pago = spark.read.format("delta").load(f"{SILVER_BASE}/shop_carrito_estados_pago")
count_estados_pago = df_estados_pago.count()
print(f"   OK {count_estados_pago:,} registros")

# 7. SHOP_FORMAPAGO (12 registros)
print("\n7. Cargando: shop_formapago")
df_formapago = spark.read.format("delta").load(f"{SILVER_BASE}/shop_formapago")
count_formapago = df_formapago.count()
print(f"   OK {count_formapago:,} registros")

total_silver = count_carrito + count_clientes + count_articulos + count_campanias + count_estados + count_estados_pago + count_formapago
print(f"\nTotal registros Silver: {total_silver:,}")
print(f"Dataset mas grande: shop_carrito ({count_carrito:,} registros)")
print(f"OBT debe tener minimo: {count_carrito:,} registros")


# =============================================================================
# PASO 2: PREPARAR DATAFRAMES PARA JOINS
# =============================================================================

print(f"\n{'='*80}")
print("PASO 2: PREPARANDO DATAFRAMES PARA JOINS")
print("="*80)

# CARRITO (base principal)
print("\nPreparando shop_carrito (tabla base)")
df_carrito_prep = df_carrito.select(
    col("id").alias("carrito_id"),
    col("fecha").alias("carrito_fecha"),
    col("monto").alias("carrito_monto"),
    col("estado").alias("carrito_estado_id"),
    col("estado_pago").alias("carrito_estado_pago_id"),
    col("id_cliente"),
    col("id_fpago"),
    col("id_campania"),
    col("localidad").alias("carrito_localidad"),
    col("provincia").alias("carrito_provincia"),
    col("anio").alias("carrito_anio"),
    col("mes").alias("carrito_mes"),
    col("dia").alias("carrito_dia"),
    col("dia_semana").alias("carrito_dia_semana"),
    col("hora").alias("carrito_hora"),
    col("segmento_monto"),
    col("monto_usd"),
    col("monto_eur"),
    col("monto_brl"),
    col("fecha_tipo_cambio")
)
print(f"   OK {df_carrito_prep.count():,} registros | {len(df_carrito_prep.columns)} columnas")

# CLIENTES
print("\nPreparando shop_clientes")
df_clientes_prep = df_clientes.select(
    col("id").alias("cliente_id"),
    col("nombre").alias("cliente_nombre"),
    col("apellido").alias("cliente_apellido"),
    col("nombre_completo").alias("cliente_nombre_completo"),
    col("email_limpio").alias("cliente_email"),
    col("telefono_limpio").alias("cliente_telefono"),
    col("sexo").alias("cliente_sexo"),
    col("sexo_desc").alias("cliente_sexo_desc"),
    col("localidad").alias("cliente_localidad"),
    col("provincia").alias("cliente_provincia"),
    col("segmento_provincia").alias("cliente_segmento_provincia"),
    col("id_pais"),
    col("pais_nombre"),
    col("pais_region"),
    col("pais_capital"),
    col("pais_poblacion"),
    col("pais_moneda")
)
print(f"   OK {df_clientes_prep.count():,} registros | {len(df_clientes_prep.columns)} columnas")

# CAMPANIAS
print("\nPreparando campanias_gp")
df_campanias_prep = df_campanias.select(
    col("id").alias("campania_id"),
    col("campania").alias("campania_nombre"),
    col("created").alias("campania_fecha_creacion"),
    col("updated").alias("campania_fecha_actualizacion"),
    col("estado").alias("campania_estado_id"),
    col("estado_desc").alias("campania_estado_desc"),
    col("anio_creacion").alias("campania_anio"),
    col("mes_creacion").alias("campania_mes")
)
print(f"   OK {df_campanias_prep.count():,} registros | {len(df_campanias_prep.columns)} columnas")

# ESTADOS CARRITO
print("\nPreparando shop_carrito_estados")
df_estados_prep = df_estados.select(
    col("id").alias("estado_carrito_id"),
    col("estado_descripcion").alias("estado_carrito_desc"),
    col("es_interno").alias("estado_carrito_es_interno"),
    col("categoria_estado").alias("estado_carrito_categoria")
)
print(f"   OK {df_estados_prep.count():,} registros | {len(df_estados_prep.columns)} columnas")

# ESTADOS PAGO
print("\nPreparando shop_carrito_estados_pago")
df_estados_pago_prep = df_estados_pago.select(
    col("id").alias("estado_pago_id"),
    col("estado_pago_desc"),
    col("nivel_riesgo").alias("estado_pago_nivel_riesgo")
)
print(f"   OK {df_estados_pago_prep.count():,} registros | {len(df_estados_pago_prep.columns)} columnas")

# FORMA PAGO
print("\nPreparando shop_formapago")
df_formapago_prep = df_formapago.select(
    col("id").alias("forma_pago_id"),
    col("forma_pago_desc"),
    col("tipo_medio").alias("forma_pago_tipo_medio")
)
print(f"   OK {df_formapago_prep.count():,} registros | {len(df_formapago_prep.columns)} columnas")


# =============================================================================
# PASO 3: CONSTRUIR OBT CON LEFT JOINS
# =============================================================================

print(f"\n{'='*80}")
print("PASO 3: CONSTRUYENDO OBT CON LEFT JOINS")
print("="*80)
print("\nEstrategia: shop_carrito como tabla base + LEFT JOINS")
print("(Garantiza que todos los 120,348 registros se mantengan)\n")

# BASE: shop_carrito
df_obt = df_carrito_prep

# JOIN 1: Clientes
print("1. JOIN: shop_carrito + shop_clientes")
df_obt = df_obt.join(df_clientes_prep, df_obt.id_cliente == df_clientes_prep.cliente_id, "left")
print(f"   OK Registros: {df_obt.count():,}")

# JOIN 2: Estados Carrito
print("\n2. JOIN: OBT + shop_carrito_estados")
df_obt = df_obt.join(df_estados_prep, df_obt.carrito_estado_id == df_estados_prep.estado_carrito_id, "left")
print(f"   OK Registros: {df_obt.count():,}")

# JOIN 3: Estados Pago
print("\n3. JOIN: OBT + shop_carrito_estados_pago")
df_obt = df_obt.join(df_estados_pago_prep, df_obt.carrito_estado_pago_id == df_estados_pago_prep.estado_pago_id, "left")
print(f"   OK Registros: {df_obt.count():,}")

# JOIN 4: Forma Pago
print("\n4. JOIN: OBT + shop_formapago")
df_obt = df_obt.join(df_formapago_prep, df_obt.id_fpago == df_formapago_prep.forma_pago_id, "left")
print(f"   OK Registros: {df_obt.count():,}")

# JOIN 5: Campanias
print("\n5. JOIN: OBT + campanias_gp")
df_obt = df_obt.join(df_campanias_prep, df_obt.id_campania == df_campanias_prep.campania_id, "left")
print(f"   OK Registros: {df_obt.count():,}")

print("\nNOTA: 'articulos' no tiene FK en shop_carrito - No incluido en OBT")
print("(En un caso real, habria una tabla shop_carrito_items con id_articulo)")


# =============================================================================
# PASO 4: AGREGAR COLUMNAS DE METADATA Y REORGANIZAR
# =============================================================================

print(f"\n{'='*80}")
print("PASO 4: FINALIZANDO OBT")
print("="*80)

# Agregar timestamp de creacion de OBT
df_obt_final = df_obt \
    .withColumn("obt_created_timestamp", current_timestamp()) \
    .withColumn("obt_created_date", current_date())

# Contar registros finales
count_obt = df_obt_final.count()
columnas_obt = len(df_obt_final.columns)

print(f"\nOBT FINAL:")
print(f"   Registros: {count_obt:,}")
print(f"   Columnas: {columnas_obt}")
print(f"   Tamano estimado: {count_obt * columnas_obt:,} valores")

# Validacion de requisito
if count_obt >= count_carrito:
    print(f"\nVALIDACION EXITOSA:")
    print(f"   OBT ({count_obt:,}) >= Dataset mas grande ({count_carrito:,})")
else:
    print(f"\nERROR: OBT tiene menos registros que el dataset mas grande!")

# Mostrar esquema
print(f"\nEsquema OBT (primeras 20 columnas):")
for i, col_name in enumerate(df_obt_final.columns[:20], 1):
    print(f"   {i:2d}. {col_name}")
if columnas_obt > 20:
    print(f"   ... y {columnas_obt - 20} columnas mas")


# =============================================================================
# PASO 5: GUARDAR OBT EN GOLD
# =============================================================================

print(f"\n{'='*80}")
print("PASO 5: GUARDANDO OBT EN GOLD")
print("="*80)

obt_path = f"{GOLD_BASE}/obt_shop_carrito"

print(f"\nGuardando OBT en: {obt_path}")
print(f"   Formato: Delta Lake")
print(f"   Modo: Overwrite")
print(f"   Compresion: Snappy")

df_obt_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("compression", "snappy") \
    .save(obt_path)

print(f"\nOBT guardado exitosamente!")


# =============================================================================
# PASO 6: VALIDACION FINAL Y ANALISIS
# =============================================================================

print(f"\n{'='*80}")
print("PASO 6: VALIDACION FINAL Y ANALISIS")
print("="*80)

# Recargar para validar
df_obt_validacion = spark.read.format("delta").load(obt_path)
count_validacion = df_obt_validacion.count()

print(f"\nValidacion de lectura:")
print(f"   Registros guardados: {count_obt:,}")
print(f"   Registros leidos: {count_validacion:,}")
print(f"   Coinciden: {'SI' if count_obt == count_validacion else 'NO'}")

# Analisis de NULLs
print(f"\nAnalisis de NULLs (primeras 10 columnas):")
for col_name in df_obt_validacion.columns[:10]:
    null_count = df_obt_validacion.filter(col(col_name).isNull()).count()
    null_pct = (null_count / count_validacion) * 100
    print(f"   {col_name:30s}: {null_count:6,} NULLs ({null_pct:5.1f}%)")

# Muestra de datos
print(f"\nMuestra de datos (5 filas, 15 columnas):")
df_obt_validacion.select(df_obt_validacion.columns[:15]).show(5, truncate=True)


# =============================================================================
# RESUMEN FINAL
# =============================================================================

print(f"\n{'='*80}")
print("RESUMEN FINAL - CAPA GOLD OBT")
print("="*80)

print(f"""
CONSIGNA EJERCICIO 4 CUMPLIDA:

TABLAS COMBINADAS:
   1. shop_carrito              ({count_carrito:,} registros) - BASE
   2. shop_clientes             ({count_clientes:,} registros) - LEFT JOIN
   3. shop_carrito_estados      ({count_estados:,} registros) - LEFT JOIN
   4. shop_carrito_estados_pago ({count_estados_pago:,} registros) - LEFT JOIN
   5. shop_formapago            ({count_formapago:,} registros) - LEFT JOIN
   6. campanias_gp              ({count_campanias:,} registros) - LEFT JOIN
   7. articulos                 ({count_articulos:,} registros) - NO JOINEABLE

OBT RESULTANTE:
   Nombre: obt_shop_carrito
   Ubicacion: {obt_path}
   Formato: Delta Lake
   Registros: {count_obt:,}
   Columnas: {columnas_obt}
   Tamano: ~{count_obt * columnas_obt:,} valores

REQUISITOS CUMPLIDOS:
   - Combina todas las tablas en 1 OBT
   - Columnas relevantes seleccionadas
   - NULLs permitidos (esperado en OBT)
   - Registros >= dataset mas grande ({count_obt:,} >= {count_carrito:,})

La OBT esta lista para ser consumida en:
{obt_path}

SIGUIENTE PASO: Documentar el pipeline completo (Ejercicio 5)
""")

StatementMeta(, 4f08306b-addf-479d-a427-fd35e8337fd9, 3, Finished, Available, Finished)

CAPA GOLD - ONE BIG TABLE (OBT)
Origen Silver: abfss://9a547f55-a8f7-4c89-b935-fb3da8226ed0@onelake.dfs.fabric.microsoft.com/4d897e0a-b68c-4840-9910-752862308e5b/Files/silver/shop_carrito
Destino Gold: abfss://9a547f55-a8f7-4c89-b935-fb3da8226ed0@onelake.dfs.fabric.microsoft.com/902f27e9-dd04-40cf-831a-2610a16fb9a7/Files/gold

Objetivo: Combinar 7 tablas Silver en 1 tabla OBT
Registros minimos esperados: 120,348 (dataset mas grande)

PASO 1: CARGANDO TABLAS SILVER

1. Cargando: shop_carrito
   OK 120,348 registros

2. Cargando: shop_clientes
   OK 115,544 registros

3. Cargando: articulos
   OK 58 registros

4. Cargando: campanias_gp
   OK 96 registros

5. Cargando: shop_carrito_estados
   OK 16 registros

6. Cargando: shop_carrito_estados_pago
   OK 5 registros

7. Cargando: shop_formapago
   OK 12 registros

Total registros Silver: 236,079
Dataset mas grande: shop_carrito (120,348 registros)
OBT debe tener minimo: 120,348 registros

PASO 2: PREPARANDO DATAFRAMES PARA JOINS

Preparand